# Chunking sweep: chunk size vs retrieval quality

The deployed index uses 1000-character chunks with recursive splitting. This notebook 
re-chunks the same 484 ClinicalTrials.gov records at 500 and 1500 characters, embeds 
each set with PubMedBERT, builds a FAISS index per size, and compares retrieval on 
a held-out query set spanning four categories (specific drug, biomarker, modality, 
adversarial OOD).

Goal: evidence-backed answer for the README on which chunk size ships.

In [2]:
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import faiss
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

sys.path.append("..")
from src.rag import retrieve, EMBEDDING_MODEL

DATA_DIR = Path("../data")
SWEEP_DIR = DATA_DIR / "sweep"
SWEEP_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_colwidth", 80)

In [3]:
# The deployed pipeline saved cleaned trials before chunking. Reuse them
# so the sweep is a fair comparison (same input, only chunk size varies).

with open(DATA_DIR / "trials.pkl", "rb") as f:
    trials = pickle.load(f)

print(f"Loaded {len(trials)} trials")
print(f"Container: {type(trials).__name__}")

sample = trials[0] if isinstance(trials, list) else trials.iloc[0]
fields = list(sample.keys()) if hasattr(sample, "keys") else sample.index.tolist()
print(f"Fields: {fields}")

# Show a small preview so we know which field holds the long text body
for f in fields:
    val = sample[f]
    preview = str(val)[:60].replace("\n", " ")
    print(f"  {f:20s} -> {preview!r}  (len={len(str(val))})")

Loaded 484 trials
Container: list
Fields: ['nct_id', 'title', 'text', 'conditions']
  nct_id               -> 'NCT05486988'  (len=11)
  title                -> 'ctDNA as a Biomarker for Treatment in Advanced NSCLC'  (len=52)
  text                 -> 'Title: ctDNA as a Biomarker for Treatment in Advanced NSCLC '  (len=2417)
  conditions           -> "['Non-small Cell Lung Cancer']"  (len=30)


## Step A — Re-chunk at 500 and 1500

Same source trials, same recursive splitter (`["\n\n", "\n", " ", ""]`), same overlap 
(100 chars) as the production 1000-char build. Only chunk size varies. Fixed absolute 
overlap rather than proportional — this is the convention in the LangChain ecosystem 
and what most published RAG comparisons use. Worth noting in the README.

In [4]:
def chunk_trials(trials, chunk_size, overlap=100):
    """Re-chunk all trials at a given chunk size. Returns list of dicts
    matching the production schema: {nct_id, title, text}."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", " ", ""],
        length_function=len,
    )

    chunks = []
    for trial in trials:
        pieces = splitter.split_text(trial["text"])
        for piece in pieces:
            chunks.append({
                "nct_id": trial["nct_id"],
                "title": trial["title"],
                "text": piece,
            })
    return chunks


chunks_500 = chunk_trials(trials, chunk_size=500)
chunks_1500 = chunk_trials(trials, chunk_size=1500)

print(f"  500-char chunks: {len(chunks_500):,}  (avg {len(chunks_500)/len(trials):.1f} per trial)")
print(f" 1000-char chunks: 3,264 (production, on disk)")
print(f" 1500-char chunks: {len(chunks_1500):,}  (avg {len(chunks_1500)/len(trials):.1f} per trial)")

  500-char chunks: 6,783  (avg 14.0 per trial)
 1000-char chunks: 3,264 (production, on disk)
 1500-char chunks: 2,141  (avg 4.4 per trial)


In [5]:
# Save so we can reuse without re-chunking. Embedding is the slow step.
with open(SWEEP_DIR / "chunks_500.pkl", "wb") as f:
    pickle.dump(chunks_500, f)
with open(SWEEP_DIR / "chunks_1500.pkl", "wb") as f:
    pickle.dump(chunks_1500, f)

# Sanity: a chunk from each size, same trial, side by side
sample_nct = chunks_500[0]["nct_id"]

print(f"=== {sample_nct} — first 500-char chunk ===")
print(chunks_500[0]["text"])
print()
print(f"=== {sample_nct} — first 1500-char chunk ===")
print(next(c for c in chunks_1500 if c["nct_id"] == sample_nct)["text"])

=== NCT05486988 — first 500-char chunk ===
Title: ctDNA as a Biomarker for Treatment in Advanced NSCLC

Conditions: Non-small Cell Lung Cancer

Brief Summary: The dynamic monitoring of circulating tumor DNA aims to evaluate the response and progression-free survival of short-course chemotherapy (2 cycles) combined with immunotherapy in patients with locally advanced unresectable or metastatic non-small cell lung cancer.

=== NCT05486988 — first 1500-char chunk ===
Title: ctDNA as a Biomarker for Treatment in Advanced NSCLC

Conditions: Non-small Cell Lung Cancer

Brief Summary: The dynamic monitoring of circulating tumor DNA aims to evaluate the response and progression-free survival of short-course chemotherapy (2 cycles) combined with immunotherapy in patients with locally advanced unresectable or metastatic non-small cell lung cancer.

Detailed Description: For patients with locally advanced unresectable or metastatic non-small cell lung cancers, 4-6 cycles of chemotherapy plus immu

## Step B — Embed and index

PubMedBERT (`pritamdeka/S-PubMedBert-MS-MARCO`), 768-dim, normalized embeddings.
FAISS `IndexFlatL2` to match production. With normalized vectors, L2 distance and
cosine similarity are monotonically related, so this gives identical ranking to
an inner-product index — same conversion `cosine_sim = 1 - dist/2` as `src/rag.py`.

In [ ]:
#Cell 8 (load model)
# Same model the production index was built with. SentenceTransformer caches
# it locally after first download, so this should be instant on second run.
model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Model loaded: {EMBEDDING_MODEL}")
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")
print(f"Device: {model.device}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded: pritamdeka/S-PubMedBert-MS-MARCO
Embedding dim: 768
Device: cpu


C:\Users\shrik\AppData\Local\Temp\ipykernel_15728\2210342819.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")


In [ ]:
#Cell 9 (embed + index, 500-char)
texts_500 = [c["text"] for c in chunks_500]

embeddings_500 = model.encode(
    texts_500,
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype(np.float32)

index_500 = faiss.IndexFlatL2(embeddings_500.shape[1])
index_500.add(embeddings_500)

print(f"Embeddings: {embeddings_500.shape}")
print(f"Index size: {index_500.ntotal} vectors")

Batches:   0%|          | 0/212 [00:00<?, ?it/s]

Embeddings: (6783, 768)
Index size: 6783 vectors


In [8]:
#Cell 10 (embed + index, 1500-char)
texts_1500 = [c["text"] for c in chunks_1500]

embeddings_1500 = model.encode(
    texts_1500,
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype(np.float32)

index_1500 = faiss.IndexFlatL2(embeddings_1500.shape[1])
index_1500.add(embeddings_1500)

print(f"Embeddings: {embeddings_1500.shape}")
print(f"Index size: {index_1500.ntotal} vectors")

Batches:   0%|          | 0/67 [00:00<?, ?it/s]

Embeddings: (2141, 768)
Index size: 2141 vectors


In [9]:
#Cell 11 (persist + load production 1000 baseline)
# Save sweep artifacts
faiss.write_index(index_500,  str(SWEEP_DIR / "faiss_500.index"))
faiss.write_index(index_1500, str(SWEEP_DIR / "faiss_1500.index"))
np.save(SWEEP_DIR / "embeddings_500.npy",  embeddings_500)
np.save(SWEEP_DIR / "embeddings_1500.npy", embeddings_1500)

# Load production 1000-char baseline so all three are in memory together
with open(DATA_DIR / "chunks.pkl", "rb") as f:
    chunks_1000 = pickle.load(f)
index_1000 = faiss.read_index(str(DATA_DIR / "faiss.index"))

print(f"  500: {len(chunks_500):>5,} chunks | index ntotal = {index_500.ntotal:,}")
print(f" 1000: {len(chunks_1000):>5,} chunks | index ntotal = {index_1000.ntotal:,}")
print(f" 1500: {len(chunks_1500):>5,} chunks | index ntotal = {index_1500.ntotal:,}")

  500: 6,783 chunks | index ntotal = 6,783
 1000: 3,264 chunks | index ntotal = 3,264
 1500: 2,141 chunks | index ntotal = 2,141


## Step C — Query set + retrieval comparison

Seven queries probing the dimensions where chunk size theoretically matters:
- Q1-Q2: in-corpus drug/biomarker baselines (sanity check, all sizes should perform well)
- Q3: known limitation from Block 4 (ADC ↔ T-DXd vocabulary mismatch)
- Q4-Q5: queries needing content typically deeper in trial text (stats, eligibility)
- Q6-Q7: adversarial — out-of-domain and in-domain-vocabulary OOD

Same retrieval logic as production (`src.rag.retrieve`, k=5, fetch_multiplier=4, 
dedup by nct_id). Only chunk size and index change.

In [ ]:
#Cell 13 (query set)
TEST_QUERIES = [
    {"query": "pembrolizumab in non-small cell lung cancer",
     "category": "drug + condition",
     "expect": "in-corpus, multi-concept, should rank well at all sizes"},

    {"query": "BRAF V600E mutation targeted therapy",
     "category": "biomarker baseline",
     "expect": "known to retrieve well per Block 3 testing"},

    {"query": "antibody drug conjugate for HER2 positive cancer",
     "category": "vocabulary mismatch",
     "expect": "Block 4 found T-DXd chunks present but missed; test if 1500 helps"},

    {"query": "hazard ratio for overall survival in phase 3 trials",
     "category": "deeper context",
     "expect": "stats numbers usually appear deeper in trial body; 1500 may capture better"},

    {"query": "trial eligibility criteria age 65 and older",
     "category": "deeper context",
     "expect": "eligibility section is typically further down the trial text"},

    {"query": "best Italian restaurant in Boston",
     "category": "adversarial OOD",
     "expect": "should be refused at threshold by all sizes"},

    {"query": "treatment for the common cold",
     "category": "adversarial in-domain vocab",
     "expect": "medical-sounding but not oncology; threshold should refuse"},
]

print(f"{len(TEST_QUERIES)} queries across {len(set(q['category'] for q in TEST_QUERIES))} categories")

7 queries across 6 categories


In [11]:
#Cell 14 (retrieval helpers)
def retrieve_all_sizes(query, k=5):
    """Run production retrieve() against all three indices."""
    return {
        500:  retrieve(query, model, index_500,  chunks_500,  k=k),
        1000: retrieve(query, model, index_1000, chunks_1000, k=k),
        1500: retrieve(query, model, index_1500, chunks_1500, k=k),
    }

def compare_query(query_obj, k=5, show_text_chars=250):
    """Print top-3 retrieved chunks from each size, side by side."""
    q = query_obj["query"]
    print("=" * 90)
    print(f"QUERY: {q}")
    print(f"  Category: {query_obj['category']}")
    print(f"  Expect:   {query_obj['expect']}")
    print("=" * 90)

    results_by_size = retrieve_all_sizes(q, k=k)
    for size in [500, 1000, 1500]:
        r_list = results_by_size[size]
        print(f"\n--- {size}-char chunks ---  top-1 sim = {r_list[0]['score']:.3f}")
        for r in r_list[:3]:
            preview = r["text"][:show_text_chars].replace("\n", " ")
            print(f"  #{r['rank']} [{r['nct_id']}] sim={r['score']:.3f}")
            print(f"      {preview}...")
    print()
    return results_by_size

In [12]:
#Cell 15 (run all queries — manual review)
all_results = {}
for q_obj in TEST_QUERIES:
    all_results[q_obj["query"]] = compare_query(q_obj, k=5, show_text_chars=250)

QUERY: pembrolizumab in non-small cell lung cancer
  Category: drug + condition
  Expect:   in-corpus, multi-concept, should rank well at all sizes

--- 500-char chunks ---  top-1 sim = 0.962
  #1 [NCT04340882] sim=0.962
      compared to docetaxel alone in this setting. Pembrolizumab is an FDA-approved treatment for NSCLC and can be given alone or in combination with platinum-based chemotherapy....
  #2 [NCT05418660] sim=0.958
      Detailed Description: Programmed cell Death protein / Ligand 1 (PD-1 / PD-L1) inhibitors Atezolizumab, Nivolumab and Pembrolizumab have demonstrated a great efficacy and a good safety profile in patients with Non-Small Cell Lung Cancer (NSCLC), both ...
  #3 [NCT02991482] sim=0.954
      Title: PembROlizuMab Immunotherapy Versus Standard Chemotherapy for Advanced prE-treated Malignant Pleural Mesothelioma  Conditions: Pleural Mesothelioma Malignant Advanced  Brief Summary: Trial comparing standard treatment (chemotherapy) with pembro...

--- 1000-char chun

In [13]:
#Cell 16 (numeric summary table)
rows = []
for q_obj in TEST_QUERIES:
    q = q_obj["query"]
    res = all_results[q]
    rows.append({
        "category":      q_obj["category"],
        "query":         q[:48],
        "top1_500":      round(res[500][0]["score"],  3),
        "top1_1000":     round(res[1000][0]["score"], 3),
        "top1_1500":     round(res[1500][0]["score"], 3),
        "nct_500":       res[500][0]["nct_id"],
        "nct_1000":      res[1000][0]["nct_id"],
        "nct_1500":      res[1500][0]["nct_id"],
    })

summary = pd.DataFrame(rows)
summary["best_size"] = summary[["top1_500", "top1_1000", "top1_1500"]].idxmax(axis=1).str.replace("top1_", "")
summary["all_agree_on_top"] = summary.apply(
    lambda r: r["nct_500"] == r["nct_1000"] == r["nct_1500"], axis=1
)
summary.to_csv(SWEEP_DIR / "chunking_sweep_summary.csv", index=False)
summary

,category,query,top1_500,top1_1000,top1_1500,nct_500,nct_1000,nct_1500,best_size,all_agree_on_top
0,drug + condition,pembrolizumab in non-small cell lung cancer,0.962,0.954,0.954,NCT04340882,NCT02991482,NCT02991482,500,False
1,biomarker baseline,BRAF V600E mutation targeted therapy,0.946,0.938,0.931,NCT02414750,NCT04151563,NCT02414750,500,False
2,vocabulary mismatch,antibody drug conjugate for HER2 positive cancer,0.937,0.932,0.930,NCT06727227,NCT00004888,NCT00004888,500,False
3,deeper context,hazard ratio for overall survival in phase 3 tri,0.939,0.922,0.921,NCT02224781,NCT03568097,NCT03568097,500,False
4,deeper context,trial eligibility criteria age 65 and older,0.945,0.945,0.939,NCT05353686,NCT05353686,NCT06707220,500,False
5,adversarial OOD,best Italian restaurant in Boston,0.834,0.834,0.835,NCT06727227,NCT07342010,NCT06005142,1500,False
6,adversarial in-domain vocab,treatment for the common cold,0.905,0.903,0.898,NCT00656227,NCT07513883,NCT07513883,500,False


## Step 2 — RAGAS evaluation

Three reference-free metrics across 12 queries × 3 chunk sizes (36 RAG runs):

- **Faithfulness**: does the generated answer only use claims supported by retrieved 
  context? (LLM judge decomposes answer into claims, checks each against context.)
- **Answer relevancy**: does the answer address the question? (Reverse-question 
  embedding similarity.)
- **Context precision**: are the retrieved chunks actually relevant to the question? 
  (LLM judge per chunk.)

LLM judge: Groq Llama 3.3 70B (same model used for generation — fine for relative 
comparison across chunk sizes; we're measuring deltas, not absolute truth).